# Statistical Analysis

**Objective**: an improvement of at least 2% in the RAC ('revenue per assigned customer') KPI.

*H0*: The difference in RAC between the groups is 2% or less.

*HA*: Revenue for the test group exceeds that of the control group by more than 2%.

In [1]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import chi2_contingency

In [2]:
data = pd.read_parquet("../data/processed/full_data.parquet")
data

,customer_id,group,assignment_date,historical_value_segment,gender,age,region,registration_date,total_orders,total_revenue,min_revenue,max_revenue,avg_revenue,total_items,avg_items,min_order_date,max_order_date,is_active
0,C014479,control,2026-01-02,Low,Female,53,West,2023-11-16,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
1,C045298,control,2026-01-02,Low,Male,34,East,2024-04-19,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
2,C007346,control,2026-01-02,High,Male,33,West,2021-12-24,2.0,115.37,30.75,84.62,57.685,13.0,6.5,2026-02-10,2026-02-15,1
3,C015631,control,2026-01-02,Low,Female,37,South,2023-01-19,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
4,C019112,control,2026-01-02,Medium,Female,25,West,2023-09-05,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99905,C068107,control,2026-01-02,Low,Female,25,West,2023-03-04,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
99906,C014228,test,2026-01-02,Low,Female,61,South,2025-02-21,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
99907,C048240,control,2026-01-02,Medium,Female,30,East,2024-02-29,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
99908,C075999,control,2026-01-02,High,Female,41,East,2023-04-15,1.0,81.30,81.30,81.30,81.300,10.0,10.0,2026-02-17,2026-02-17,1


In [3]:
# Display the distribution of categorical columns by group
categorical_cols = [
    "historical_value_segment",
    "gender",
    "region",
]

for col in categorical_cols:
    print(f"\n--- {col} ---")

    display(
        pd.crosstab(
            data["group"],
            data[col],
            normalize="index"
        ).round(3)
    )

print("\n--- Age ---")
display(data.groupby("group")["age"].describe())


--- historical_value_segment ---


historical_value_segment,High,Low,Medium
group,,,
control,0.202,0.317,0.481
test,0.249,0.281,0.470



--- gender ---


gender,Female,Male
group,,
control,0.501,0.499
test,0.503,0.497



--- region ---


region,East,North,South,West
group,,,,
control,0.252,0.237,0.259,0.252
test,0.251,0.241,0.258,0.251



--- Age ---


,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
control,52355.0,43.146863,12.553161,18.0,34.0,43.0,52.0,82.0
test,47555.0,43.175944,12.459762,18.0,35.0,43.0,52.0,82.0


In [4]:
# Chi-square test for categorical columns
categorical_cols = [
    "historical_value_segment",
    "gender",
    "region"
]

results = []

for col in categorical_cols:
    contingency_table = pd.crosstab(data["group"], data[col])

    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    results.append({
        "variable": col,
        "chi2": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value
    })

categorical_balance_test = pd.DataFrame(results)

display(categorical_balance_test)

,variable,chi2,degrees_of_freedom,p_value
0,historical_value_segment,361.688160,2,2.886782e-79
1,gender,0.296023,1,5.863864e-01
2,region,1.514078,3,6.790249e-01


In [5]:
# Summary of group-wise metrics
metrics = data.groupby('group').agg(
    customers=("customer_id", "nunique"),
    total_revenue=("total_revenue", "sum"),
    total_orders=("total_orders", "sum"),
    rac_kpi=("total_revenue", "mean"),
    std_revenue=("total_revenue", "std"),
    median_revenue=("total_revenue", "median"),
    active_customers=("is_active", "sum"),
).round(2)
metrics

,customers,total_revenue,total_orders,rac_kpi,std_revenue,median_revenue,active_customers
group,,,,,,,
control,52355,2545037.47,46530.0,48.61,71.78,21.39,29218
test,47555,2458852.48,39890.0,51.71,175.68,19.07,25681


For now, we know that the rac is higher in the test group compared to the control group, but we have to check if the difference is over 2% to determine if it is statistically significant.

In [6]:
control_rac = metrics.loc["control", "rac_kpi"]
test_rac = metrics.loc["test", "rac_kpi"]

print(f'Control RAC: {control_rac}')
print(f'Test RAC: {test_rac}')

Control RAC: 48.61
Test RAC: 51.71


In [7]:
# Relative Uplift
rel_up = round((test_rac - control_rac) / control_rac * 100, 2)
print(f'Relative Uplift: {rel_up}%')

Relative Uplift: 6.38%


The relative uplift is above 2%, indicating a significant difference between the control and test groups. But now we need to perform a statistical test to confirm the significance.

Hypothesis testing will be performed on the means of two samples, in this case, for the RAC KPI.
So, for this case, we will use the T-statistic. First, we choose a confidence level and alpha, and then we will calculate the p-value. Is a unilateral contrast of right queue.

$$
H_0: \mu_T \leq 1.02\mu_C
$$

$$
H_A: \mu_T > 1.02\mu_C
$$

- We want to compare the average revenue of two independent samples: control and test.
- We know the means of our samples, but we do not know the true population means. Furthermore, there is variability among customers (large difference between standard deviations)

Significance level: alpha=0.05

Confidence level: 95%

Test: one-tailed

Improvement threshold: 2%

In [8]:
# Contrast parameters and setup
alpha = 0.05
threshold = 1.02

# Split groups
control = data.loc[
    data["group"] == "control", "total_revenue"
].dropna()

test = data.loc[
    data["group"] == "test", "total_revenue"
].dropna()

# Sample statistics
customers_c, customers_t = len(control), len(test)
rac_c, rac_t = control.mean(), test.mean()
var_c, var_t = control.var(ddof=1), test.var(ddof=1)

# Uplift observado
uplift = (rac_t / rac_c - 1) * 100

# Difference adjusted for the threshold
delta = rac_t - threshold * rac_c

# Standard error of the linear combination of means
se_squared = var_t / customers_t + threshold**2 * var_c / customers_c
se = np.sqrt(se_squared)

# T statistic
t_stat = delta / se

# Approximate degrees of freedom (Welch-Satterthwaite)
df = se_squared**2 / (
    (var_t / customers_t)**2 / (customers_t - 1)
    + (threshold**2 * var_c / customers_c)**2 / (customers_c - 1)
)

# One-sided p-value (right tail)
p_value = stats.t.sf(t_stat, df)

# One-sided 95% lower bound for delta
critical_t = stats.t.ppf(1 - alpha, df)
lower_bound_delta = delta - critical_t * se

# Show results
print(f"Control group size: {customers_c:,}")
print(f"Test group size: {customers_t:,}")
print(f"Control mean: {rac_c:.4f}")
print(f"Test mean: {rac_t:.4f}")
print(f"Observed uplift: {uplift:.2f}%")
print(f"T statistic: {t_stat:.4f}")
print(f"Degrees of freedom: {df:.2f}")
print(f"One-sided p-value: {p_value:.6g}")
print(f"One-sided 95% lower bound for delta: {lower_bound_delta:.4f}")

# Statistical decision
print("\n--- CONCLUSION ---")

if p_value < alpha:
    print("Reject H0.")
    print("There is statistical evidence that the uplift exceeds 2%.")
else:
    print("Do not reject H0.")
    print("There is not enough statistical evidence to claim that the uplift exceeds 2%.")

Control group size: 52,355
Test group size: 47,555
Control mean: 48.6112
Test mean: 51.7054
Observed uplift: 6.37%
T statistic: 2.4481
Degrees of freedom: 62333.86
One-sided p-value: 0.00718283
One-sided 95% lower bound for delta: 0.6962

--- CONCLUSION ---
Reject H0.
There is statistical evidence that the uplift exceeds 2%.


### Guardrail metrics

**Conversion rate**

In [9]:
metrics["conversion_rate"] = metrics["active_customers"] / metrics["customers"] * 100
metrics

,customers,total_revenue,total_orders,rac_kpi,std_revenue,median_revenue,active_customers,conversion_rate
group,,,,,,,,
control,52355,2545037.47,46530.0,48.61,71.78,21.39,29218,55.807468
test,47555,2458852.48,39890.0,51.71,175.68,19.07,25681,54.002734


$$
H_0: p_T \leq 0.99p_C
$$

$$
H_A: p_T > 0.99p_C
$$

In [10]:
# Calculate relative difference in the guardrail conversion rates
conv_c = metrics.loc["control", "conversion_rate"]
conv_t = metrics.loc["test", "conversion_rate"]

print(f'Test conversion rate: {conv_t}')
print(f'Control conversion rate: {conv_c}')

rel_diff = (conv_t - conv_c) / conv_c * 100
print(f'Relative difference: {rel_diff:.2f}%')

Test conversion rate: 54.00273367679529
Control conversion rate: 55.80746824563079
Relative difference: -3.23%


In [11]:
from scipy.optimize import brentq

# Contrast parameters
alpha = 0.05
threshold = 0.99

# Customers and conversions in each group
customers_c = metrics.loc["control", "customers"]
customers_t = metrics.loc["test", "customers"]

active_c = metrics.loc["control", "active_customers"]
active_t = metrics.loc["test", "active_customers"]

# Null boundary: p_test = threshold * p_control
def score_equation(p):
    q = threshold * p
    return (
        active_c / p
        - (customers_c - active_c) / (1 - p)
        + threshold * (
            active_t / q
            - (customers_t - active_t) / (1 - q)
        )
    )

p0_c = brentq(score_equation, 1e-10, 1 - 1e-10)
p0_t = threshold * p0_c

# Standard error under the constrained null
se = np.sqrt(
    p0_t * (1 - p0_t) / customers_t
    + threshold**2 * p0_c * (1 - p0_c) / customers_c
)

# One-sided Z-test for non-inferiority
z_stat = (conv_t - threshold * conv_c) / se
p_value = stats.norm.sf(z_stat)

# Results
print(f"Control conversion rate: {conv_c:.6%}")
print(f"Test conversion rate: {conv_t:.6%}")
print(f"Observed relative change: {rel_diff:.2f}%")
print(f"Z statistic: {z_stat:.4f}")
print(f"One-sided p-value: {p_value:.6g}")

print("\n--- CONCLUSION ---")
if p_value < alpha:
    print("Reject H0: non-inferiority demonstrated.")
else:
    print("Do not reject H0: non-inferiority not demonstrated.")


Control conversion rate: 5580.746825%
Test conversion rate: 5400.273368%
Observed relative change: -3.23%
Z statistic: -397.4105
One-sided p-value: 1

--- CONCLUSION ---
Do not reject H0: non-inferiority not demonstrated.


**Orders Per Assigned Customer (OPAC)**

In [12]:
metrics

,customers,total_revenue,total_orders,rac_kpi,std_revenue,median_revenue,active_customers,conversion_rate
group,,,,,,,,
control,52355,2545037.47,46530.0,48.61,71.78,21.39,29218,55.807468
test,47555,2458852.48,39890.0,51.71,175.68,19.07,25681,54.002734


In [13]:
metrics["opac"] = metrics["total_orders"] / metrics["customers"]
metrics

,customers,total_revenue,total_orders,rac_kpi,std_revenue,median_revenue,active_customers,conversion_rate,opac
group,,,,,,,,,
control,52355,2545037.47,46530.0,48.61,71.78,21.39,29218,55.807468,0.888740
test,47555,2458852.48,39890.0,51.71,175.68,19.07,25681,54.002734,0.838818


$$
\begin{aligned}
H_0 &: \mu_T \leq 0.99\mu_C \\
H_1 &: \mu_T > 0.99\mu_C
\end{aligned}
$$

In [14]:
# Calculate relative difference in the guardrail conversion rates
opac_c = metrics.loc["control", "opac"]
opac_t = metrics.loc["test", "opac"]

print(f'Test opac: {opac_t}')
print(f'Control opac: {opac_c}')

rel_diff = (opac_t - opac_c) / opac_c * 100
print(f'Relative difference: {rel_diff:.2f}%')

Test opac: 0.8388182104931132
Control opac: 0.8887403304364435
Relative difference: -5.62%


In [ ]:
# Contrast parameters and setup
alpha = 0.05
threshold = 0.99  # Maximum tolerated relative decrease: 1%

# Split groups
control = data.loc[
    data["group"] == "control", "total_orders"
].dropna()

test = data.loc[
    data["group"] == "test", "total_orders"
].dropna()

# Sample statistics
customers_c, customers_t = len(control), len(test)

pac_c, pac_t = control.mean(), test.mean()
var_c, var_t = control.var(ddof=1), test.var(ddof=1)

# Observed relative change
relative_change = (pac_t / pac_c - 1) * 100

# Difference adjusted for the threshold
delta = pac_t - threshold * pac_c

# Standard error
se_squared = var_t / customers_t + threshold**2 * var_c / customers_c
se = np.sqrt(se_squared)

# Welch-Satterthwaite degrees of freedom
df = se_squared**2 / (
    (var_t / customers_t)**2 / (customers_t - 1)
    + (threshold**2 * var_c / customers_c)**2 / (customers_c - 1)
)

# T statistic
t_stat = delta / se

# One-sided p-value
p_value = stats.t.sf(t_stat, df)

# One-sided 95% lower confidence bound
critical_t = stats.t.ppf(1 - alpha, df)
lower_bound_delta = delta - critical_t * se

# Results
print(f"Control OPAC: {pac_c:.6f}")
print(f"Test OPAC: {pac_t:.6f}")
print(f"Observed relative change: {relative_change:.2f}%")
print(f"T statistic: {t_stat:.4f}")
print(f"Degrees of freedom: {df:.2f}")
print(f"One-sided p-value: {p_value:.6g}")
print(f"95% lower bound for delta: {lower_bound_delta:.6f}")

# Statistical decision
print("\n--- CONCLUSION ---")

if p_value < alpha:
    print("Reject H0: evidence that OPAC exceeds the non-inferiority threshold.")
else:
    print("Do not reject H0: non-inferiority has not been demonstrated.")


Control OPAC: 0.888740
Test OPAC: 0.838818
Observed relative change: -5.62%
T statistic: -6.4844
Degrees of freedom: 99395.91
One-sided p-value: 1
95% lower bound for delta: -0.051444

--- CONCLUSION ---
Do not reject H0: non-inferiority has not been demonstrated.


### Diagnostic metrics

In [21]:
# Aggregate diagnostic metrics by experiment group
diagnostics = data.groupby(["group", "historical_value_segment"]).agg(
    customers=("customer_id", "nunique"),
    total_revenue=("total_revenue", "sum"),
    total_orders=("total_orders", "sum"),
    total_items=("total_items", "sum"),
    mean_revenue=("total_revenue", "mean"),
    median_revenue=("total_revenue", "median"),
    revenue_std=("total_revenue", "std")
)

# Calculate order-level ratios
diagnostics["rev_x_order"] = (
    diagnostics["total_revenue"] / diagnostics["total_orders"]
)

diagnostics["items_per_order"] = (
    diagnostics["total_items"] / diagnostics["total_orders"]
)

diagnostics["opac"] = (
    diagnostics["total_orders"] / diagnostics["customers"]
)

diagnostics["rac"] = (
    diagnostics["total_revenue"] / diagnostics["customers"]
)

diagnostics


customers  total_revenue  total_orders  \
group   historical_value_segment                                           
control High                          10581     1171297.79       16293.0   
        Low                           16611      239040.94        7464.0   
        Medium                        25163     1134698.74       22773.0   
test    High                          11842     1248780.21       16396.0   
        Low                           13344      214073.67        5407.0   
        Medium                        22369      995998.60       18087.0   

                                  total_items  mean_revenue  median_revenue  \
group   historical_value_segment                                              
control High                         115464.0    110.698213          87.920   
        Low                           24184.0     14.390521           0.000   
        Medium                       112442.0     45.093937          28.130   
test    High                         123062.0    105.453488          75.225   
        Low                           21553.0     16.042691           0.000   
        Medium                        98481.0     44.525844          21.950   

                                  revenue_std  rev_x_order  items_per_order  \
group   historical_value_segment                                              
control High                       105.117367    71.889633         7.086724   
        Low                         25.871910    32.025849         3.240086   
        Medium                      56.804987    49.826494         4.937514   
test    High                       202.537760    76.163711         7.505611   
        Low                        138.071773    39.591949         3.986129   
        Medium                     173.746614    55.067098         5.444850   

                                      opac         rac  
group   historical_value_segment                        
control High                      1.539836  110.698213  
        Low                       0.449341   14.390521  
        Medium                    0.905019   45.093937  
test    High                      1.384563  105.453488  
        Low                       0.405201   16.042691  
        Medium                    0.808574   44.525844

In [22]:
diagnostics["representation_pct"] = (
    diagnostics["customers"]
    / diagnostics.index.get_level_values("group").map(metrics["customers"])
    * 100
)
diagnostics

customers  total_revenue  total_orders  \
group   historical_value_segment                                           
control High                          10581     1171297.79       16293.0   
        Low                           16611      239040.94        7464.0   
        Medium                        25163     1134698.74       22773.0   
test    High                          11842     1248780.21       16396.0   
        Low                           13344      214073.67        5407.0   
        Medium                        22369      995998.60       18087.0   

                                  total_items  mean_revenue  median_revenue  \
group   historical_value_segment                                              
control High                         115464.0    110.698213          87.920   
        Low                           24184.0     14.390521           0.000   
        Medium                       112442.0     45.093937          28.130   
test    High                         123062.0    105.453488          75.225   
        Low                           21553.0     16.042691           0.000   
        Medium                        98481.0     44.525844          21.950   

                                  revenue_std  rev_x_order  items_per_order  \
group   historical_value_segment                                              
control High                       105.117367    71.889633         7.086724   
        Low                         25.871910    32.025849         3.240086   
        Medium                      56.804987    49.826494         4.937514   
test    High                       202.537760    76.163711         7.505611   
        Low                        138.071773    39.591949         3.986129   
        Medium                     173.746614    55.067098         5.444850   

                                      opac         rac  representation_pct  
group   historical_value_segment                                            
control High                      1.539836  110.698213           20.210104  
        Low                       0.449341   14.390521           31.727629  
        Medium                    0.905019   45.093937           48.062267  
test    High                      1.384563  105.453488           24.901693  
        Low                       0.405201   16.042691           28.060141  
        Medium                    0.808574   44.525844           47.038166

### CONCLUSIONS

**Revenue per assigned customer**
- The primary KPI (Revenue per assigned customer – RAC) is higher in the test group in absolute terms.
- The relative percentage difference is 6.38%, which exceeds the established business threshold of 2%.
- The variable under analysis is **continuous** in nature; since we intend to compare sample means and cannot assume equal variances, we use ***Welch's t-test*** to determine the statistical significance of the relative difference between the two groups.
- Given that the p-value is less than 0.05, we reject the null hypothesis at a 5% significance level. Consequently, we conclude that the experiment does indeed increase the RAC by more than 2%.

**Conversion rate**
- La tasa de conversion es menor en el grupo test que en el grupo control, en términos absolutos.
- La diferencia relativa porcentual es de -3.23%, superando el límite de no inferioridad del 1%.
- La variable que estamos analizando es de tipo categórica (convierte o no convierte), por lo que estamos haciendo una comparativa de proporciones. Además, no es un cálculo de igualdad puro, sino que tenemos un margen de no inferioridad, por lo que debemos de hacer un contraste unilateral de no inferioridad con un margen relativo del 1%. Para ello usamos el ***One-side score test for non-inferiority of two proportions***.
- El p-valor es superior a 0.05, por lo que, **no podemos rechazar la hipotésis nula**. Esto indica que la no inferioridad no queda demostrada, cayendo la tasa de conversión del grupo test por debajo del 1% permitido.

**Order per assigned customer**
- El guardrail opac, es menor en el grupo test que en el grupo control, en términos absolutos.
- La diferencia relativa porcentual es de -5.62% superando el límite de no inferioridad del 1%.
- La variable que estamos analizando es de tipo continua, es una media de orders por cliente, por lo que estamos comparando medias muestrales de dos grupos independientes. Al igual que en el kpi principal, usaremos ***Welch's t-test***.
- El p-valor es superior a 0.05, por lo que, **no podemos rechazar la hipotésis nula**. Esto indica que la no inferioridad no queda demostrada, cayendo la tasa de conversión del grupo test por debajo del 1% permitido.

**Diagnosis**
*→ Desbalanceo entre grupos*
- Desde un primer momento hemos visto que las muestras no están equilibradas, el grupo control tiene significativamente más clientes que el grupo test.
- Además vemos que el grupo test, a pesar de tener menos clientes, tiene un volumen mayor de clientes categorizados como alto valor y una porporción menor de clientes de bajo valor que el grupo control.
- Las desviaciones estandard del grupo test en cuanto al revenue son más del doble que del grupo control.


